# 01. Análise descritiva (EDA)

Exploração visual de `registro_unificado` — missing, nomes, sexo, data de nascimento, CEP, UF.



In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

%matplotlib inline

from config import FILTRO_UF, USE_PHONETIC_STRIP_VOWELS, get_connection, print_paths, require_tables
from eda import (
    cep_prefix_top, count_by_origem, dob_year_distribution, missingness_table,
    nome_meio_rate, plot_cep_prefix, plot_dob_year_hist, plot_missingness_bars,
    plot_origem_counts, plot_sexo_distribution, plot_top_bars, plot_uf_distribution,
    sexo_distribution, top_values, uf_distribution,
)

print_paths()
print('FILTRO_UF:', FILTRO_UF)
con = get_connection()
require_tables(con, ['registro_unificado'], notebook_origem='00')

## 1. Visão geral


In [ ]:
df_origem = count_by_origem(con)
display(df_origem)
plot_origem_counts(df_origem)

## 2. Missingness


In [ ]:
EDA_COLS = [
    'nome_completo', 'primeiro_nome', 'nome_meio', 'ultimo_nome', 'nome_mae',
    'sexo', 'data_nascimento', 'cep', 'uf', 'cpf_norm',
    'nome_completo_phon', 'primeiro_nome_phon', 'ultimo_nome_phon',
]
if USE_PHONETIC_STRIP_VOWELS:
    EDA_COLS += ['nome_completo_phon_sv', 'primeiro_nome_phon_sv', 'ultimo_nome_phon_sv']

df_missing = missingness_table(con, columns=EDA_COLS)
display(df_missing.pivot(index='coluna', columns='origem', values='pct_missing').round(1))
plot_missingness_bars(df_missing)

## 3. Nomes


In [ ]:
for col, titulo in [('primeiro_nome', 'Primeiro nome'), ('ultimo_nome', 'Último nome')]:
    df_top = top_values(con, col, n=20)
    display(df_top.head(10))
    plot_top_bars(df_top, title=f'Top 20 — {titulo}')

display(nome_meio_rate(con))

## 4. Sexo


In [ ]:
df_sexo = sexo_distribution(con)
display(df_sexo)
plot_sexo_distribution(df_sexo)

## 5. Data de nascimento


In [ ]:
df_dob = dob_year_distribution(con)
display(df_dob.groupby('origem')['n'].sum().reset_index(name='com_ano_valido'))
plot_dob_year_hist(df_dob)

## 6. CEP


In [ ]:
df_cep = cep_prefix_top(con, top_n=15)
display(df_cep.head(20))
plot_cep_prefix(df_cep)

## 7. UF


In [ ]:
df_uf = uf_distribution(con)
display(df_uf.head(15))
plot_uf_distribution(df_uf)

## 8. Nome da mãe


In [ ]:
df_mae_miss = missingness_table(con, columns=['nome_mae'])
display(df_mae_miss)
df_mae_top = top_values(con, 'nome_mae', n=15)
plot_top_bars(df_mae_top, title='Top 15 — nome da mãe')

## 9. Resumo

Revise acima:
- **Missingness** — colunas críticas para linkage (nome, DOB, CEP, nome_mae)
- **Nomes** — concentração nos top tokens (blocking)
- **CEP/UF** — coerência com `FILTRO_UF`


In [ ]:
con.close()